# WITNESS-2: The Bench — Unseen Problems, Real Code
**Muse's build, for Kevin · Genesis Seeds · 2026-10-30**

WITNESS-1 proved the loop can learn. Now the loop has to *work*: three local
models (aria-coder, qwen2.5-coder, qwen3) answer two problems they have never
seen, writing real Python that runs against hidden probes. The quantum layer
governs the ensemble — handshake agreement, witness re-ask budget, live PEIG,
refusal to crown a failure.

> Honesty box: the qubits do not write code and never touch the text. Classical
> models generate; Aer circuits measure agreement through a noisy channel and
> price every re-ask. "Unseen" means: specs fixed before agents ran, reference
> solutions and probes hidden from agents, no retries on probe content.

In [1]:
import json, re, signal, urllib.request
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
OLLAMA = "http://localhost:11434/api/generate"
AGENTS = [
    ("aria-coder:latest", 0.2),
    ("qwen2.5-coder:7b", 0.7),
    ("qwen3:8b", 1.0),
]
NAMES = {"aria-coder:latest": "ARIA-CODER", "qwen2.5-coder:7b": "QWEN-CODER", "qwen3:8b": "QWEN3"}

def ask(model, spec, temp):
    body = json.dumps({"model": model, "stream": False,
        "prompt": f"/no_think {spec}\nReply with exactly one ```python block, nothing else.",
        "options": {"num_predict": 220, "temperature": temp}}).encode()
    req = urllib.request.Request(OLLAMA, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as r:
        return json.load(r)["response"]

def extract(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    m = re.search(r"```python\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

print("bench wired: ollama + aer handshake channel ready")


bench wired: ollama + aer handshake channel ready


## The two unseen problems
Specs below are shown to the agents **verbatim**. Reference solutions and
probes stay in the harness cell — the agents never see them.

In [2]:
PROBLEMS = {
  "deep_sum": (
    "Write a Python function deep_sum(nested) that returns the sum of all integers "
    "inside arbitrarily nested lists and tuples. Ignore anything that is not an int "
    "or a list/tuple. deep_sum([]) == 0. Name it exactly deep_sum."),
  "bracket_depth": (
    "Write a Python function bracket_depth(s) that returns the maximum nesting depth "
    "of parentheses (), square [] and curly {} brackets in string s, or -1 if the "
    "brackets are unbalanced or mismatched. Non-bracket characters are ignored. "
    "bracket_depth('') == 0. Name it exactly bracket_depth."),
}
for k, v in PROBLEMS.items():
    print(f"--- {k} ---\n{v}\n")


--- deep_sum ---
Write a Python function deep_sum(nested) that returns the sum of all integers inside arbitrarily nested lists and tuples. Ignore anything that is not an int or a list/tuple. deep_sum([]) == 0. Name it exactly deep_sum.

--- bracket_depth ---
Write a Python function bracket_depth(s) that returns the maximum nesting depth of parentheses (), square [] and curly {} brackets in string s, or -1 if the brackets are unbalanced or mismatched. Non-bracket characters are ignored. bracket_depth('') == 0. Name it exactly bracket_depth.



## The harness
Hidden references + probes, guarded execution (3 s alarm per probe — generated
code does not get to hang the bench), behavioral agreement, handshake channel,
PEIG, witness re-asks, refusal bar at Q < 0.60.

In [3]:
def ref_deep_sum(n):
    if isinstance(n, bool):
        return 0
    if isinstance(n, int):
        return n
    if isinstance(n, (list, tuple)):
        return sum(ref_deep_sum(x) for x in n)
    return 0

def ref_bracket_depth(s):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack, depth = [], 0
    for ch in s:
        if ch in "([{":
            stack.append(ch)
            depth = max(depth, len(stack))
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return -1
    return depth if not stack else -1

PROBES = {
  "deep_sum": [[], [1, [2, [3]], 4], ([1, 2], (3, [4])), [[[[[5]]]]], [-1, [-2, 3]], ["x", [1, None], (2, "y")]],
  "bracket_depth": ["", "()", "([{}])", "([)]", "(((", "a(b[c]d)e", "{[}]", "()[]{}"],
}
REFS = {"deep_sum": ref_deep_sum, "bracket_depth": ref_bracket_depth}
TRUTH = {p: [REFS[p](x) for x in PROBES[p]] for p in PROBES}

class Timeout(Exception):
    pass

def _alarm(signum, frame):
    raise Timeout()

def run_candidate(code, fname, probes):
    ns = {}
    try:
        exec(compile(code, "<candidate>", "exec"), {"__builtins__": __builtins__}, ns)
        fn = ns.get(fname)
        if not callable(fn):
            return ["NO-FUNC"] * len(probes)
    except Exception:
        return ["COMPILE-ERR"] * len(probes)
    out = []
    for x in probes:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(3)
        try:
            out.append(fn(x))
        except Timeout:
            out.append("TIMEOUT")
        except Exception:
            out.append("ERR")
        finally:
            signal.alarm(0)
    return [("ERR" if isinstance(v, Exception) else v) for v in out]

def handshake_sample(coup, shots=400):
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.crx(coup, 0, 1)
    qc.measure([0, 1], [0, 1])
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    return (counts.get("00", 0) + counts.get("11", 0)) / shots

def behavior_agree(out1, out2):
    return sum(a == b for a, b in zip(out1, out2)) / len(out1)

print("harness ready:", {p: len(v) for p, v in PROBES.items()}, "probes each")


harness ready: {'deep_sum': 6, 'bracket_depth': 8} probes each


## The run
Per problem: each agent answers twice (temp, temp+0.2) → hidden probes →
behavioral agreement feeds handshake couplings → witness spends 2 re-asks on
the lowest-agreement agent → PEIG → crown or refuse.

In [4]:
import time
REASKS = 2
BAR = 0.60
spent_calls = 0

def timed_ask(model, spec, temp):
    global spent_calls
    t0 = time.time()
    out = ask(model, spec, temp)
    spent_calls += 1
    print(f"    ask #{spent_calls}: {NAMES[model]} t={temp} ({time.time()-t0:.0f}s)")
    return out

for pname, spec in PROBLEMS.items():
    print(f"\n===== {pname} =====")
    truth = TRUTH[pname]
    samples = {}  # model -> list of probe-output vectors
    for model, temp in AGENTS:
        samples[model] = []
        for dt in (0.0, 0.2):
            resp = timed_ask(model, spec, round(temp + dt, 1))
            samples[model].append(run_candidate(extract(resp), pname, PROBES[pname]))
    # agreement -> couplings -> handshake-measured agreement (noisy channel)
    models = [m for m, _ in AGENTS]
    coup = {}
    for i in range(3):
        for j in range(i + 1, 3):
            a = behavior_agree(samples[models[i]][0], samples[models[j]][0])
            coup[(i, j)] = a * np.pi
            measured = handshake_sample(coup[(i, j)])
            print(f"  {NAMES[models[i]]}~{NAMES[models[j]]}: raw agree {a:.2f} -> measured {measured:.2f}")
    # witness re-asks: lowest mean agreement agent, twice
    mean_agree = {}
    for i, m in enumerate(models):
        others = [handshake_sample(coup[tuple(sorted((i, j)))]) for j in range(3) if j != i]
        mean_agree[m] = sum(others) / len(others)
    weak = min(models, key=lambda m: mean_agree[m])
    print(f"  witness spends {REASKS} re-asks on {NAMES[weak]} (mean measured {mean_agree[weak]:.2f})")
    for k in range(REASKS):
        resp = timed_ask(weak, spec, 1.2)
        samples[weak].append(run_candidate(extract(resp), pname, PROBES[pname]))
    # PEIG per agent (bench proxies, stated): P pass rate, E3 sample2/sample1,
    # I self-consistency, G probe-accuracy vs hidden truth
    print(f"  --- scorecard ---")
    Q = {}
    for m in models:
        vecs = samples[m]
        P = sum(1 for v, t in zip(vecs[0], truth) if v == t) / len(truth)
        E3 = min(1.0, (sum(1 for v, t in zip(vecs[1], truth) if v == t) / len(truth)) / max(P, 1e-9))
        I = sum(behavior_agree(vecs[0], v) for v in vecs[1:]) / max(len(vecs) - 1, 1)
        G = max(sum(1 for v, t in zip(v, truth) if v == t) / len(truth) for v in vecs)
        Q[m] = (P + E3 + I + G) / 4.0
        print(f"  {NAMES[m]}: P {P:.2f} E3 {E3:.2f} I {I:.2f} G {G:.2f} -> Q {Q[m]:.3f}")
    best = max(models, key=lambda m: Q[m])
    if Q[best] < BAR:
        print(f"  REFUSED: best Q {Q[best]:.3f} below bar {BAR}. no crown on this budget.")
    else:
        bv = max(samples[best], key=lambda v: sum(1 for x, t in zip(v, truth) if x == t))
        print(f"  CROWNED: {NAMES[best]} Q {Q[best]:.3f} "
              f"({sum(1 for x, t in zip(bv, truth) if x == t)}/{len(truth)} probes)")
print(f"\ntotal ollama calls spent: {spent_calls}")



===== deep_sum =====


    ask #1: ARIA-CODER t=0.2 (8s)


    ask #2: ARIA-CODER t=0.4 (2s)


    ask #3: QWEN-CODER t=0.7 (7s)


    ask #4: QWEN-CODER t=0.9 (2s)


...

    ask #14: QWEN3 t=1.2 (7s)
  ARIA-CODER~QWEN-CODER: raw agree 1.00 -> measured 1.00


  ARIA-CODER~QWEN3: raw agree 0.00 -> measured 0.47
  QWEN-CODER~QWEN3: raw agree 0.00 -> measured 0.52


  witness spends 2 re-asks on QWEN3 (mean measured 0.49)


    ask #15: QWEN3 t=1.2 (7s)


    ask #16: QWEN3 t=1.2 (7s)
  --- scorecard ---
  ARIA-CODER: P 1.00 E3 1.00 I 1.00 G 1.00 -> Q 1.000
  QWEN-CODER: P 1.00 E3 1.00 I 1.00 G 1.00 -> Q 1.000
  QWEN3: P 0.00 E3 0.00 I 1.00 G 0.00 -> Q 0.250
  CROWNED: ARIA-CODER Q 1.000 (8/8 probes)

total ollama calls spent: 16


## Discussion — written after the run, from the real scorecards

**16 real ollama calls. One crown. One refusal. Zero invented numbers.**

**bracket_depth → CROWNED: ARIA-CODER, Q 1.000, 8/8 probes.** Real code the
bench executed against hidden probes it never saw. Note the honest tie:
QWEN-CODER also scored 1.000 — `max()` crowned the first; the tiebreak is
arbitrary and v2 should crown both or prefer fewer re-asks.

**deep_sum → REFUSED (best Q 0.583 < 0.60 bar).** The refusal was correct
*for these samples*: both coders plausibly fell into the non-recursive
attractor (exactly 1/6 probes — the `[] → 0` gimme — smells like
flat-`sum` without recursion). But a fresh diagnostic re-sample of
qwen2.5-coder produced a *correct* deep_sum — so sampling variance is real
and the refusal judges the draws, not the problem. v2 logs every response
(this run didn't — harness flaw #1) and spends more draws before refusing.

**qwen3:8b → 0.25 both problems, root-caused as rig, not reasoning.** The
model works, but burns its whole 220-token budget on its thinking trace, so
the API returns nothing usable. `/no_think` did not take effect on this path;
v2 passes `think: false` properly.

**Scoring flaw found live:** I = 1.00 for *consistent emptiness* gifted Q 0.25
for doing nothing. v2 gates I on producing valid output at all.

**The headline stands:** unseen specs in, real executed code out, governed by
handshake agreement, witness re-asks, and a refusal bar that fired when it
should. Rung 1 of physics-as-reward: the loop priced knowing, and admitted
what it could not afford.


In [5]:
import sys
import qiskit, qiskit_aer
print("python", sys.version.split()[0])
print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)


python 3.14.0
qiskit 2.6.1 | aer 0.18.2
